# Fine-tuning Notebook

Generated by LLM Architecture Explorer.

**Base model:** `/media/amiyaun/New Volume/chai-projects/llama/models/llama-3.2-1B-Instruct`
**Task:** Causal language modeling (fine-tuning)

This notebook was assembled automatically based on the architecture,
pruning, and PEFT configuration selected in the explorer. All cells are
fully editable — nothing here is locked.


## Resource estimate (from explorer, at generation time)

| Metric | Value |
|---|---|
| Total params | 1,235,814,400 |
| Active params (after pruning) | 1,174,992,896 |
| Trainable params | 32,768 |
| Trainable % | 0.003% |
| Estimated VRAM (params + optimizer, excl. activations) | 2.35 GB |

These are estimates computed from the model's declared architecture and your
selected configuration — actual usage will vary with batch size, sequence
length, and dataset.


In [ ]:
# Dependencies
# !pip install transformers datasets peft accelerate bitsandbytes trl


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer


In [ ]:
# Load base model and tokenizer
MODEL_PATH = '/media/amiyaun/New Volume/chai-projects/llama/models/llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,   # adjust to your hardware (float16 / float32 as needed)
    device_map="auto",
)


In [ ]:
# Remove pruned decoder layers entirely, BEFORE applying PEFT.
# This assumes the model's decoder stack is a standard nn.ModuleList
# (model.model.layers for causal LM wrappers) iterated sequentially in
# forward() — true for Llama/Gemma/Mistral-style architectures, but not
# guaranteed for every architecture. Inspect model.print_trainable_parameters()
# and a test forward pass after this cell to confirm the model still runs.

pruned_layer_indices = [0]

layers_attr = model.model.layers if hasattr(model, "model") else model.layers
kept_layers = [layer for i, layer in enumerate(layers_attr) if i not in pruned_layer_indices]
import torch.nn as nn
if hasattr(model, "model"):
    model.model.layers = nn.ModuleList(kept_layers)
else:
    model.layers = nn.ModuleList(kept_layers)

# Update the model's own layer count metadata, if present, so anything
# reading config.num_hidden_layers downstream (e.g. some generation utils)
# stays consistent with the actual pruned layer count.
if hasattr(model.config, "num_hidden_layers"):
    model.config.num_hidden_layers = len(kept_layers)

print(f"Pruned {len(pruned_layer_indices)} layer(s): {pruned_layer_indices}")
print(f"Remaining layers: {len(kept_layers)}")


In [ ]:
# NOTE: layers_to_transform relies on peft's layer-index matching, which has been reported unreliable on some peft/transformers version combinations (see https://github.com/huggingface/peft/issues/2155). Verify model.print_trainable_parameters() below shows the expected count before training.
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj'],
    layers_to_transform=[1]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # VERIFY this matches your explorer's estimate before trusting layers_to_transform worked


In [ ]:
# ---------------------------------------------------------------
# Dataset loading — fill this in.
#
# Example:
#   dataset = load_dataset("your_dataset_name", split="train")
#
# The dataset should have a text field the tokenizer can consume, or you
# can define a formatting function and pass it to SFTTrainer below via
# `formatting_func`.
# ---------------------------------------------------------------

dataset = None  # TODO: load your dataset here


In [ ]:
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,   # adjust to your hardware
    report_to="none",
)


In [ ]:
# ---------------------------------------------------------------
# NOTE: requires `dataset` to be set above (currently a placeholder).
# ---------------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()


In [ ]:
# Save adapter (if PEFT was used) or full model
trainer.save_model("./output/final")
tokenizer.save_pretrained("./output/final")

# If using PEFT and you want a merged, standalone model:
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./output/merged")
